In [2]:
import cygnet
import torch, time

test_fixed_precision = False

def total_loss(pred, target):
    return cygnet.focal_loss(pred, target)

cygnet.start_timer()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
torch.set_default_device(device)

b_size = 5
num_batches = 2
epochs = 5

H = 1152
W = 2048

data = [torch.randn(b_size, 1, H, W) for _ in range(num_batches)]
targets = [torch.randn(b_size, 1, H, W) for _ in range(num_batches)]
dummy = torch.randn(1, 1, H, W)

cygnet.end_timer_and_print("setup")
print("\n=======\n")
cygnet.start_timer()

net = cygnet.Net()
opt = torch.optim.SGD(net.parameters(), lr=0.001)
scaler = torch.amp.GradScaler("cuda", enabled=True)

b = 0
for epoch in range(epochs):
    print(f"start epoch {epoch}:")
    for input, target in zip(data, targets):
        print(f"batch #{b}")
        b += 1
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
            output = net(input)
            loss = total_loss(output, target)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
cygnet.end_timer_and_print("Mixed precision:")

net.eval()
# Warmup
for _ in range(10):
    _ = net(dummy)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
        _ = net(dummy)
torch.cuda.synchronize()
tim = (time.perf_counter() - t0) / 10 * 1000
print(f"in inference: {tim:.1f} ms per frame")

Using cuda device

setup
Total execution time = 0.003 sec
Max memory used by tensors = 0.356 Gbytes


start epoch 0:
batch #0
batch #1
start epoch 1:
batch #2
batch #3
start epoch 2:
batch #4
batch #5
start epoch 3:
batch #6
batch #7
start epoch 4:
batch #8
batch #9

Mixed precision:
Total execution time = 3.608 sec
Max memory used by tensors = 8.464 Gbytes
in inference: 21.1 ms per frame
